# Assignment 2 — Customer Segmentation with KMeans, DBSCAN, and PCA
### Credit Card Customer Behavior Dataset

**Goal:** Group customers by behavior (spending, payments, cash advances, credit usage) using two
different clustering algorithms, and understand *how* each one arrives at its groups rather than
assuming every cluster it finds is automatically a real, meaningful business segment.

I'm treating this the same way I treated Assignment 1: every step gets explained before I run it —
what I did, what I'm seeing, and what I'll do next — not just a wall of code.

**Dataset:** [Credit Card Dataset for Clustering](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata)
(`CC GENERAL.csv`) — ~8,950 credit card customers, 17 behavioral/numeric features per customer
(balance, purchases, cash advances, credit limit, payments, tenure, and more). No target column at
all, because this is unsupervised — there's nothing to predict, only structure to discover.

**Why this dataset fits the brief well:**
- It's explicitly transactional/behavioral (exactly the domain the brief asks for).
- Nearly every column is already a meaningful numeric feature — I don't have to invent one.
- It's big enough (~8,950 rows) that clusters found in it are more than noise from a tiny sample.
- It has real missing values in a couple of columns, so the "handle missing/invalid values" step is
  a genuine decision, not a formality.

> **Before running:** download `CC GENERAL.csv` from the Kaggle link above and place it at
> `data/CC_GENERAL.csv`. See the README for exact steps.


## 1. Imports and Setup

Same reasoning as Assignment 1 — I group imports by purpose and fix a `RANDOM_SEED` everywhere
that randomness is involved (KMeans' centroid initialization, mainly), so the notebook produces the
same clusters every time it's re-run.

New this time: `sklearn.cluster` for KMeans and DBSCAN, `sklearn.decomposition` for PCA, and
`sklearn.metrics` for `silhouette_score` — the metric I'll use to judge cluster quality without a
ground-truth label to check against (there isn't one; that's the nature of unsupervised learning).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded.")


## 2. Load the Dataset

Same rule as before: look at the raw data before touching it.


In [ ]:
DATA_PATH = "data/CC_GENERAL.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 3. Inspect Feature Distributions and Missing Values

Per the brief, I check dtypes, missing values, and get a feel for each feature's distribution
before deciding anything about scaling or feature selection.


In [ ]:
print("=== Dtypes ===")
print(df.dtypes)

print("\n=== Missing values per column ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== Duplicate rows ===")
print(df.duplicated().sum())


**What I'm seeing:** `CREDIT_LIMIT` has 1 missing value and `MINIMUM_PAYMENTS` has a few hundred
missing. These aren't data-entry typos — a missing `MINIMUM_PAYMENTS` most plausibly means a
customer never carried a balance that required a minimum payment, and a missing `CREDIT_LIMIT` is
almost certainly a single data-processing gap given it's just 1 row out of ~8,950. Either way, I
need a decision before scaling, because `StandardScaler` and every distance-based algorithm below
will fail outright on a `NaN`.


In [ ]:
summary_stats = df.describe().T
summary_stats["missing"] = df.isnull().sum()
summary_stats[["mean", "std", "min", "50%", "max", "missing"]]


**Reading the distributions:** columns like `PURCHASES`, `CASH_ADVANCE`, and `PAYMENTS` have
means far above their medians (50th percentile) — a classic sign of a right-skewed distribution
with a handful of very high-spending customers pulling the average up. This matters later: KMeans
uses Euclidean distance, so a few extreme outliers in an unscaled feature can single-handedly
distort where cluster centers end up. I'll come back to this once I've selected my final features.


## 4. Select Meaningful Numerical Features & Remove Identifiers

**Removing identifiers:** `CUST_ID` is a unique string per customer with zero behavioral meaning —
if left in as a feature it would need to be dropped anyway (it's non-numeric), but conceptually the
important point from the brief is that identifiers should never influence a distance calculation.
An ID doesn't describe *behavior*, so it doesn't belong in the feature set at all.

**Selecting features:** the raw dataset has 17 numeric columns. Rather than throwing all of them
in, I picked a smaller set that each describe a genuinely different axis of customer behavior —
this keeps the clusters interpretable, and avoids near-duplicate columns (e.g.
`ONEOFF_PURCHASES` + `INSTALLMENTS_PURCHASES` already sum to `PURCHASES`, so including all three
would let "purchasing" dominate the distance calculation three times over).

| Feature | What it captures |
|---|---|
| `BALANCE` | How much the customer currently owes |
| `PURCHASES` | Total amount spent on purchases |
| `CASH_ADVANCE` | Total cash withdrawn against the card (a different, often riskier behavior than purchasing) |
| `CREDIT_LIMIT` | How much credit the bank has extended them |
| `PAYMENTS` | Total amount they've paid back |
| `PURCHASES_FREQUENCY` | How often they purchase (0 = never, 1 = very regularly) — frequency, not just amount |
| `TENURE` | How many months they've held the card — a proxy for account age |
| `AVG_ORDER_VALUE` *(engineered)* | `PURCHASES / PURCHASES_TRX` — the brief specifically lists this as an example feature, and it isn't in the raw data, so I derive it myself: how much they spend *per purchase*, not just in total |

That's 8 features across spending, payment behavior, credit exposure, and account age — different
enough from each other that clusters found across them should mean something.


In [ ]:
# Derived feature: average order value = total purchase amount / number of purchase transactions.
# Guard against divide-by-zero for customers who made zero purchase transactions.
df["AVG_ORDER_VALUE"] = np.where(df["PURCHASES_TRX"] > 0, df["PURCHASES"] / df["PURCHASES_TRX"], 0)

selected_features = [
    "BALANCE", "PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT",
    "PAYMENTS", "PURCHASES_FREQUENCY", "TENURE", "AVG_ORDER_VALUE",
]

cluster_df = df[selected_features].copy()
print("Selected feature set:", selected_features)
cluster_df.describe().T


## 5. Handle Missing and Invalid Values

Only `CREDIT_LIMIT` is missing within my selected feature set (1 row) — `MINIMUM_PAYMENTS` isn't
one of my chosen features, so its missingness doesn't affect me here. I use **median imputation**
rather than mean, specifically because I already noticed these financial columns are right-skewed —
the median is far less sensitive to the extreme high-spending outliers than the mean is, so it's a
more representative "typical" value to fill in.

I also check for **invalid values** — negative balances or purchases wouldn't make sense for this
data and would signal a data-quality problem, not real customer behavior.


In [ ]:
print("Missing CREDIT_LIMIT rows before imputation:", cluster_df["CREDIT_LIMIT"].isnull().sum())

median_credit_limit = cluster_df["CREDIT_LIMIT"].median()
cluster_df["CREDIT_LIMIT"] = cluster_df["CREDIT_LIMIT"].fillna(median_credit_limit)

print("Missing CREDIT_LIMIT rows after imputation:", cluster_df["CREDIT_LIMIT"].isnull().sum())

print("\n=== Check for invalid (negative) values ===")
print((cluster_df < 0).sum())


No negative values anywhere — good, the data is internally consistent. No missing values remain
either. The feature set is now clean and fully numeric, ready for scaling.


## 6. Scale the Features

**Why scaling matters here — this is the most important preprocessing decision in the whole
notebook.** Both KMeans and DBSCAN decide which points belong together using **distance**
(Euclidean distance, specifically). Look at the raw ranges: `CREDIT_LIMIT` can run into the tens of
thousands, while `PURCHASES_FREQUENCY` is bounded between 0 and 1. If I clustered on raw values,
`CREDIT_LIMIT` alone would dominate every distance calculation — two customers with wildly
different purchase habits but similar credit limits would look "close together" to the algorithm,
purely because of the units the columns happen to be measured in, not because they actually behave
similarly. Scaling puts every feature on the same footing (mean 0, standard deviation 1) so the
algorithm judges similarity based on *behavioral pattern*, not arbitrary units.

I use `StandardScaler` rather than min-max scaling because several of these features have real
outliers (a few customers with very high cash advances or balances) — standardizing around the
mean and standard deviation handles that more gracefully than squeezing everything into a fixed
[0, 1] range, which would compress the bulk of ordinary customers into a tiny sliver of the range
just to accommodate a few extreme values.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)
X_scaled_df = pd.DataFrame(X_scaled, columns=selected_features)

print("Before scaling (raw units):")
print(cluster_df.describe().loc[["mean", "std"]].round(2))

print("\nAfter scaling (mean should be ~0, std should be ~1 for every feature):")
print(X_scaled_df.describe().loc[["mean", "std"]].round(2))


Every feature now has mean ≈ 0 and standard deviation ≈ 1 — none of them can structurally
dominate a distance calculation anymore just because of its original units. This is the input both
KMeans and DBSCAN will actually cluster on from here forward.


## 7. KMeans — Finding K with the Elbow Method

KMeans needs to be told how many clusters (`K`) to look for up front — it doesn't discover that
number on its own. So the first job is figuring out a *reasonable* K, not just picking one
arbitrarily.

**Inertia** is the sum of squared distances from each point to its own cluster's center — it always
goes down as K increases (more clusters means points are, on average, closer to *some* center), so
I can't just pick the K with the lowest inertia, or I'd end up picking the largest K available every
time. Instead I look for the **elbow** — the point where adding another cluster stops buying much
of a reduction in inertia. Past that point, I'm mostly just splitting genuinely similar customers
into artificially separate groups.

I run KMeans across a range of K values and record inertia for each one, exactly as the brief asks.


In [ ]:
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)
    print(f"K={k:2d}  ->  inertia={km.inertia_:.1f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), inertia_values, marker="o")
ax.set_xlabel("K (number of clusters)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method: Inertia vs K")
plt.tight_layout()
plt.savefig("outputs/kmeans_elbow.png")
plt.show()


## 8. KMeans — Silhouette Score

The elbow plot is useful but subjective — "where the bend happens" is a judgment call, and on real
data the bend is rarely as clean as a textbook diagram. **Silhouette score** gives me a second,
more numeric opinion: for every point, it compares how close that point is to others in its own
cluster versus the nearest *other* cluster. It ranges from -1 (probably in the wrong cluster) to +1
(well-matched to its own cluster and clearly separated from others). Averaged across all points, it
gives one number per K that I can directly compare.

I calculate it for the same range of K values as the elbow plot, so I can cross-check the two
methods against each other rather than trusting either one alone.


In [ ]:
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k:2d}  ->  silhouette={score:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), silhouette_scores, marker="o", color="darkorange")
ax.set_xlabel("K (number of clusters)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score vs K")
plt.tight_layout()
plt.savefig("outputs/kmeans_silhouette.png")
plt.show()


## 9. Choose Final K and Profile Each Cluster

**How I'm choosing K:** I look for a value that sits near the elbow bend *and* has a
comparatively strong silhouette score — not necessarily the single highest silhouette score, since
the very highest silhouette often belongs to a very small K (like K=2) that barely splits the data
into anything useful. I want the smallest K that still captures a real bend in inertia while keeping
a reasonably healthy silhouette score, since that gives the most *interpretable* segmentation
without needlessly fragmenting genuinely similar customers.

`FINAL_K` below is intentionally left as a variable I set explicitly, once I've looked at both
plots together, rather than automating the choice — picking K is a judgment call informed by the
numbers, not a formula.


In [ ]:
# I set this after reading the elbow + silhouette plots above together.
FINAL_K = 4

kmeans_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_SEED, n_init=10)
cluster_df["kmeans_cluster"] = kmeans_final.fit_predict(X_scaled)

print(f"Final K = {FINAL_K}")
print(f"Inertia at this K: {kmeans_final.inertia_:.1f}")
print(f"Silhouette score at this K: {silhouette_score(X_scaled, cluster_df['kmeans_cluster']):.4f}")
print("\nCluster sizes:")
print(cluster_df["kmeans_cluster"].value_counts().sort_index())


**Cluster profile table** — the required descriptive statistics per cluster. I look at the mean
of every original (unscaled) feature within each cluster, so the numbers below are in real, readable
units (dollars, months, frequency) rather than the standardized values the algorithm actually used
internally.


In [ ]:
cluster_profile = cluster_df.groupby("kmeans_cluster")[selected_features].mean().round(2)
cluster_profile["count"] = cluster_df["kmeans_cluster"].value_counts().sort_index()
cluster_profile.to_csv("outputs/kmeans_cluster_profile.csv")
cluster_profile


In [ ]:
# A visual companion to the table above -- easier to compare clusters at a glance
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, feat in enumerate(selected_features):
    sns.boxplot(data=cluster_df, x="kmeans_cluster", y=feat, ax=axes[i], palette="Set2")
    axes[i].set_title(feat)
plt.tight_layout()
plt.savefig("outputs/kmeans_cluster_boxplots.png")
plt.show()


## 10. DBSCAN — Density-Based Clustering

**Why DBSCAN doesn't need K in advance:** unlike KMeans, which starts by picking K center points
and assigning everyone to their nearest one, DBSCAN works by growing clusters outward from dense
neighborhoods of points — it keeps absorbing neighboring points as long as there are enough of them
close together (`min_samples` within radius `eps`), and stops when the density runs out. However
many dense regions it finds is however many clusters it reports. That also means it can naturally
say "this point doesn't belong to any dense region" and label it as **noise**, something KMeans
structurally can't do — KMeans is forced to assign every single point to some cluster, even a clear
outlier.

**The trade-off:** DBSCAN trades away the need to pre-specify K for needing two *different*
parameters instead — `eps` (how close points must be to count as neighbors) and `min_samples` (how
many neighbors a point needs to be considered part of a dense region). Getting those two right is
its own tuning problem, which is what the experiments below are actually testing.


In [ ]:
from sklearn.neighbors import NearestNeighbors

# A common heuristic for picking a starting eps: plot the distance to each point's k-th nearest
# neighbor (k = min_samples), sorted ascending, and look for the "knee" in the curve -- similar
# in spirit to the KMeans elbow plot, but for density instead of centroid distance.
k_neighbors = 8
neighbors = NearestNeighbors(n_neighbors=k_neighbors)
neighbors_fit = neighbors.fit(X_scaled)
distances, _ = neighbors_fit.kneighbors(X_scaled)
k_distances = np.sort(distances[:, k_neighbors - 1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_distances)
ax.set_xlabel("Points, sorted by distance")
ax.set_ylabel(f"Distance to {k_neighbors}th nearest neighbor")
ax.set_title("K-Distance Plot (helps pick a starting eps for DBSCAN)")
plt.tight_layout()
plt.savefig("outputs/dbscan_kdistance.png")
plt.show()


**Required experiment: multiple eps/min_samples configurations.** I don't just eyeball one
setting from the plot above and stop — I sweep a small grid of `eps` and `min_samples` combinations
around what that plot suggests, and record how many clusters and how much noise each one produces,
so I can see directly how sensitive DBSCAN is to its parameters.


In [ ]:
dbscan_results = []
eps_values = [0.5, 0.8, 1.0, 1.5, 2.0]
min_samples_values = [5, 8, 12]

for eps in eps_values:
    for min_samples in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_scaled)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        noise_pct = n_noise / len(labels) * 100

        # Silhouette score only makes sense with 2+ clusters and needs noise points excluded
        if n_clusters >= 2:
            mask = labels != -1
            sil = silhouette_score(X_scaled[mask], labels[mask]) if mask.sum() > 1 else np.nan
        else:
            sil = np.nan

        dbscan_results.append({
            "eps": eps, "min_samples": min_samples,
            "n_clusters": n_clusters, "n_noise": n_noise,
            "noise_pct": round(noise_pct, 1), "silhouette": sil,
        })

dbscan_results_df = pd.DataFrame(dbscan_results)
dbscan_results_df.to_csv("outputs/dbscan_grid_search.csv", index=False)
dbscan_results_df


**What I'm looking for in this table:** DBSCAN is noticeably more sensitive to its parameters
than KMeans is to K. A small `eps` tends to produce many tiny clusters plus a lot of noise (the
radius is too tight to link up genuinely similar customers), while a large `eps` tends to merge
everything into one or two giant blobs (the radius is loose enough that dissimilar customers get
absorbed into the same "dense" region). I pick my final configuration from whichever row balances a
reasonable number of clusters against a sensible (not enormous) noise percentage and a healthy
silhouette score.


In [ ]:
# I set this after reading the grid search table above.
FINAL_EPS = 1.0
FINAL_MIN_SAMPLES = 8

dbscan_final = DBSCAN(eps=FINAL_EPS, min_samples=FINAL_MIN_SAMPLES)
cluster_df["dbscan_cluster"] = dbscan_final.fit_predict(X_scaled)

n_clusters_final = len(set(cluster_df["dbscan_cluster"])) - (1 if -1 in cluster_df["dbscan_cluster"].values else 0)
n_noise_final = int((cluster_df["dbscan_cluster"] == -1).sum())

print(f"Final DBSCAN config: eps={FINAL_EPS}, min_samples={FINAL_MIN_SAMPLES}")
print(f"Clusters found: {n_clusters_final}")
print(f"Points labeled as noise: {n_noise_final} ({n_noise_final / len(cluster_df) * 100:.1f}% of all customers)")
print("\nCluster sizes (label -1 = noise):")
print(cluster_df["dbscan_cluster"].value_counts().sort_index())


**Why the noise points are useful, not just discarded:** the customers DBSCAN labels `-1` aren't
"bad data" — they're customers whose behavior doesn't fit neatly into any dense, common pattern.
In a real business setting, these are often the most operationally interesting rows: unusually
high spenders, unusually erratic account activity, or genuinely novel behavior that an
average-based summary would blur into a nearby cluster. KMeans, by contrast, would have been forced
to assign every one of these unusual customers to whichever centroid happened to be nearest, quietly
hiding them inside an otherwise "normal" segment.


In [ ]:
# A quick look at who actually got flagged as noise, compared to the overall population
noise_profile = cluster_df[cluster_df["dbscan_cluster"] == -1][selected_features].mean().round(2)
overall_profile = cluster_df[selected_features].mean().round(2)

comparison = pd.DataFrame({"noise_mean": noise_profile, "overall_mean": overall_profile})
comparison["ratio"] = (comparison["noise_mean"] / comparison["overall_mean"]).round(2)
comparison


## 11. PCA — Reducing to Two Dimensions for Visualization

My feature space has 8 dimensions — I can't directly plot that. PCA finds new axes (principal
components), each a weighted combination of the original 8 features, ordered so the first component
captures as much of the overall variance in the data as possible, the second captures as much of
the *remaining* variance as possible, and so on. Taking just the first two components gives me a
2D snapshot that preserves as much of the original structure as two dimensions possibly can — not
a perfect picture, but the best flat approximation available.

I apply PCA to the **scaled** features, not the raw ones — if I skipped scaling here, PCA would
have the exact same problem KMeans and DBSCAN would: `CREDIT_LIMIT`'s huge raw range would
dominate the variance calculation and the first component would basically just *be* credit limit,
rather than a genuine blend of behavior.


In [ ]:
pca_full = PCA(random_state=RANDOM_SEED)
pca_full.fit(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var), start=1):
    print(f"PC{i}: explains {ev*100:5.2f}% individually, {cv*100:5.2f}% cumulative")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, len(explained_var) + 1), explained_var, alpha=0.7, label="Individual")
ax.plot(range(1, len(explained_var) + 1), cumulative_var, marker="o", color="darkred", label="Cumulative")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title("PCA Explained Variance")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/pca_explained_variance.png")
plt.show()


In [ ]:
# Reduce to exactly 2 components for visualization
pca_2d = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca_2d.fit_transform(X_scaled)

pc1_var, pc2_var = pca_2d.explained_variance_ratio_
print(f"PC1 explains {pc1_var*100:.2f}% of variance")
print(f"PC2 explains {pc2_var*100:.2f}% of variance")
print(f"Together, these 2 components explain {(pc1_var + pc2_var)*100:.2f}% of the original 8-feature variance")


**Important caveat, stated plainly:** whatever percentage the two components add up to, that's
also exactly how much structure from the original 8-dimensional space is **not** shown in the 2D
plots below. Two points that land close together in the PCA plot are close *along the two
directions of greatest variance* — they could still be genuinely different in some dimension the
first two components didn't prioritize. The scatter plots below are a useful, honest summary, not a
complete picture of every relationship in the data.


## 12. Visualize KMeans Clusters in PCA Space


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_df["kmeans_cluster"],
                      cmap="Set2", s=15, alpha=0.7)
ax.set_xlabel(f"PC1 ({pc1_var*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pc2_var*100:.1f}% variance)")
ax.set_title(f"KMeans Clusters (K={FINAL_K}) in PCA Space")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.tight_layout()
plt.savefig("outputs/kmeans_pca_scatter.png")
plt.show()


## 13. Visualize DBSCAN Clusters and Noise in PCA Space

Noise points (label `-1`) are plotted separately in a neutral color so they're visually
distinguishable from actual clusters, rather than just becoming "another color in the legend."


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

noise_mask = cluster_df["dbscan_cluster"] == -1
cluster_mask = ~noise_mask

scatter = ax.scatter(X_pca[cluster_mask, 0], X_pca[cluster_mask, 1],
                      c=cluster_df.loc[cluster_mask, "dbscan_cluster"],
                      cmap="Set2", s=15, alpha=0.7, label="Clustered")
ax.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1],
           c="lightgray", s=15, alpha=0.5, marker="x", label="Noise")

ax.set_xlabel(f"PC1 ({pc1_var*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pc2_var*100:.1f}% variance)")
ax.set_title(f"DBSCAN Clusters + Noise (eps={FINAL_EPS}, min_samples={FINAL_MIN_SAMPLES}) in PCA Space")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/dbscan_pca_scatter.png")
plt.show()


## 14. KMeans vs DBSCAN — Direct Comparison

Side by side, so the shape difference between the two algorithms' clusters is immediately visible
in the same PCA space.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_df["kmeans_cluster"], cmap="Set2", s=12, alpha=0.7)
axes[0].set_title(f"KMeans (K={FINAL_K})")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(X_pca[cluster_mask, 0], X_pca[cluster_mask, 1],
                c=cluster_df.loc[cluster_mask, "dbscan_cluster"], cmap="Set2", s=12, alpha=0.7)
axes[1].scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], c="lightgray", s=12, alpha=0.5, marker="x")
axes[1].set_title(f"DBSCAN (eps={FINAL_EPS}, min_samples={FINAL_MIN_SAMPLES})")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.savefig("outputs/kmeans_vs_dbscan_pca.png")
plt.show()


**What to actually compare here:** KMeans clusters are built to be roughly round/convex around
a centroid by construction — that's a structural property of how the algorithm works, not a
reflection of the data's true shape. DBSCAN clusters can be any shape, since they just follow
wherever the density leads, and it's the only one of the two that can call some points noise
instead of forcing them into a group. Whether that flexibility actually finds a *better*
segmentation for this specific dataset, or just fragments things further, is exactly the trade-off
to weigh once real numbers are in hand.


## 15. Wrap-Up

All outputs are saved under `outputs/`:
- `kmeans_elbow.png`, `kmeans_silhouette.png` — K selection evidence
- `kmeans_cluster_profile.csv`, `kmeans_cluster_boxplots.png` — cluster profile table + visuals
- `dbscan_kdistance.png`, `dbscan_grid_search.csv` — DBSCAN parameter tuning evidence
- `pca_explained_variance.png` — how much variance each component explains
- `kmeans_pca_scatter.png`, `dbscan_pca_scatter.png`, `kmeans_vs_dbscan_pca.png` — cluster visuals

These are exactly what's needed to answer the **Required Observations** in the brief (how scaling
changed the result, how K was picked, what distinguishes each KMeans cluster, how sensitive DBSCAN
was to its parameters, what the noise points look like, how much variance the first two PCA
components captured, and which algorithm was more useful here) — that write-up happens in
`observations.md` once the real run's numbers are in, since the actual answers depend on what this
notebook produces on your machine, not on assumptions made ahead of time.
